# 11 · E5: one combined adapter vs three specialists (Step 3.6)

**Runtime: L4 GPU** (Runtime → Change runtime type → **L4 GPU**).

1. **Train** one adapter on all three jobs at once: 51,626 examples (13,134 answering + 17,967 answerability + 20,525 quality), shuffled together, 1 epoch = 3,227 optimizer steps. Each example keeps its own task instruction, which tells the adapter which job to do. Same recipe as the specialists (D-051, D-052).
2. **Evaluate** it on all three test sets.
3. **Compare** with the three specialist adapters (E2) and the untrained model (E1), on the same examples.

Expect about **11 hours of training, so 2–3 Colab sessions**, plus ~45 minutes of evaluation. That is expected: the run saves every 10 minutes.

**When Colab disconnects or the session ends:** reconnect (L4) and **Run all** again. Training continues from the last checkpoint; evaluation only starts once training has finished, and finished tasks are skipped.

Every command runs through `sh()`, which stops with an error if the command fails.

## 1 · Settings, Drive, code, W&B key

In [ ]:
import json
import os
import subprocess

from google.colab import drive, userdata

REPO_URL = "https://github.com/Vishwapujara/vizwiz-assistant.git"
BRANCH = "main"
REPO_DIR = "/content/vizwiz-assistant"
DRIVE_ROOT = "/content/drive/MyDrive/vizwiz"
DATA = f"{DRIVE_ROOT}/datasets"
LOCAL = "/content/data"
CKPT = f"{DRIVE_ROOT}/checkpoints"
RUNS = f"{DRIVE_ROOT}/runs"
VENV = "/content/venv"
PY = f"{VENV}/bin/python"
RUN_ID = "e5-multi-lora"
ADAPTER = f"{CKPT}/{RUN_ID}/adapter"
TASKS = ("answer", "answerability", "quality")
SPECIALIST = {t: f"{RUNS}/e2-{t}-lora" for t in TASKS}


def sh(cmd, check=True, **env):
    """Run a shell command, stream its output, and raise if it fails (unless check=False)."""
    proc = subprocess.Popen(
        cmd,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env={**os.environ, **env},
    )
    for line in proc.stdout:
        print(line, end="")
    code = proc.wait()
    if check and code:
        raise RuntimeError(f"Command failed (exit {code}). Stop here and paste the output above.")
    return code


try:
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")  # never printed
    print("W&B key loaded from Colab Secrets.")
except Exception as e:
    raise RuntimeError(
        "Add WANDB_API_KEY in the key icon (Secrets) on the left, turn on notebook access, then rerun."
    ) from e
os.environ["WANDB_PROJECT"] = "vizwiz-assistant"
os.environ["HF_HOME"] = "/content/hf_cache"

drive.mount("/content/drive")
sh(
    f"if [ -d {REPO_DIR}/.git ]; then git -C {REPO_DIR} fetch -q origin {BRANCH} && git -C {REPO_DIR} reset -q --hard origin/{BRANCH}; else git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}; fi"
)
sh(f"git -C {REPO_DIR} log --oneline -1")
sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")
ckpts = (
    sorted(d for d in os.listdir(f"{CKPT}/{RUN_ID}") if d.startswith("checkpoint-"))
    if os.path.isdir(f"{CKPT}/{RUN_ID}")
    else []
)
print("saved checkpoints so far:", ckpts or "none (fresh start)")

**Paste back:** the commit line, the GPU line (NVIDIA L4), and the `saved checkpoints so far` line.

## 2 · Pinned environment (~3–5 min)

In [ ]:
%%time
sh("pip install -q uv")
sh(f"uv venv -q --clear --python 3.13 {VENV}")
sh(
    f"uv pip install -q --python {PY} -e {REPO_DIR} -r {REPO_DIR}/requirements/colab-gpu.txt -c {REPO_DIR}/requirements/colab-gpu.lock.txt"
)
sh(f"{PY} -m vizwiz.compat versions")

## 3 · Unzip photos (~4 min)

In [ ]:
%%time
import hashlib

expected = json.load(open(f"{REPO_DIR}/results/splits/files.json"))
for name in ("vqa_test.jsonl", "quality_test.jsonl"):
    digest = hashlib.sha256(open(f"{DATA}/processed/{name}", "rb").read()).hexdigest()
    assert digest == expected[name]["sha256"], f"{name} differs from the committed hash"
    print(f"{name}: matches committed hash")
sh(f"{PY} -m vizwiz.data.download extract --root {DATA} --dest {LOCAL} --splits train val")

## 4 · Train (~11 h in total, across sessions)
The first model download (~9 GB) happens here. The first line says `starting fresh` or `resuming from checkpoint-N`. Val loss on 500 mixed val examples every 400 steps; live curve in W&B (run `e5-multi-lora`).

If training already finished, this cell is skipped.

In [ ]:
%%time
summary_path = f"{CKPT}/{RUN_ID}/train_summary.json"
if os.path.exists(summary_path) and os.path.exists(f"{ADAPTER}/adapter_config.json"):
    print("training already finished: skipped")
else:
    sh(
        f"{PY} -m vizwiz.train.sft train --config {REPO_DIR}/configs/train/multi.yaml --mode lora --records {DATA}/processed --images {LOCAL}/images --out {CKPT} --run-id {RUN_ID} --wandb --wandb-group e5"
    )
s = json.load(open(summary_path))
print(
    "TRAIN DONE:",
    {
        k: s[k]
        for k in (
            "global_step",
            "optimizer_steps_planned",
            "resumed_from",
            "train_loss",
            "final_val_loss",
            "examples_per_s",
            "peak_mem_gb",
            "wall_s",
        )
    },
)

**Paste back:** the `TRAIN DONE: ...` line (or, if the session ended mid-training, the last progress line before it stopped).

## 5 · Evaluate on all three test sets (~45 min)
One vLLM start, then answering, answerability and quality. Finished tasks are kept in `metrics.json` and skipped on a rerun.

In [ ]:
%%time
OUT = f"{RUNS}/{RUN_ID}"
done = (
    set(json.load(open(f"{OUT}/metrics.json"))["tasks"])
    if os.path.exists(f"{OUT}/metrics.json")
    else set()
)
todo = [t for t in TASKS if t not in done]
if not todo:
    print("evaluation already done: skipped")
else:
    sh(
        f"{PY} -m vizwiz.eval.run_eval --records {DATA}/processed --images {LOCAL}/images --split test --tasks {' '.join(todo)} --adapter {ADAPTER} --run-id {RUN_ID} --out {OUT}"
    )

**Paste back:** the three result lines (`[answer] vqa_accuracy`, `[answerability] average_precision`, `[quality] macro_f1`).

## 6 · Compare: untrained vs specialists vs combined (~2 min)
Paired on the same examples; the reference is the specialist (E2), so the E5 difference reads "combined minus specialist".

In [ ]:
for t in TASKS:
    sh(
        f"{PY} -m vizwiz.eval.compare --records {DATA}/processed --split test --task {t} --runs {RUNS}/e1-baseline {SPECIALIST[t]} {OUT} --names e1 e2 e5 --reference e2 --report {OUT}/compare_{t}.json > /dev/null"
    )
    c = json.load(open(f"{OUT}/compare_{t}.json"))
    print(
        f"COMPARE {t} {c['metric']} n={c['n']}:",
        {k: (r["value"], r["diff_vs_reference"], r["diff_ci95"]) for k, r in c["runs"].items()},
    )
m = json.load(open(f"{OUT}/metrics.json"))["tasks"]
ab = m["answerability"]["metrics"]
print(
    "E5 answerability: recall@P0.9",
    ab["recall_at_precision_0.9"],
    " invalid %",
    ab["invalid_output_pct"],
)
print(
    "E5 quality per-label F1:",
    {k: v["f1"] for k, v in m["quality"]["metrics"]["per_label"].items()},
)
print("E5 quality JSON valid %:", m["quality"]["metrics"]["json_valid_pct"])

**Paste back:** everything this cell printed (numbers only).

## 7 · Flush Drive, then disconnect
Upload back (summary numbers only):
- `MyDrive/vizwiz/checkpoints/e5-multi-lora/train_summary.json`
- `MyDrive/vizwiz/checkpoints/e5-multi-lora/settings.json`
- `MyDrive/vizwiz/runs/e5-multi-lora/metrics.json`
- `MyDrive/vizwiz/runs/e5-multi-lora/compare_answer.json`, `compare_answerability.json`, `compare_quality.json`

Then Runtime → **Disconnect and delete runtime**.

Keep `checkpoints/e5-multi-lora/adapter/` on Drive. The `checkpoint-*` folders (~0.8 GB) can be deleted once the numbers look right.

In [ ]:
drive.flush_and_unmount()
print("Drive flushed. Safe to disconnect.")